# DAViD Sketch Notebook

This notebook orchestrates the training pipeline. All logic lives in:
- `config.py`: paths, hyperparameters
- `dataset.py`: `OptimizedVideoDataset`, data loaders
- `model.py`: `ClassificationHead`, `SwiGLU`, transforms
- `encoder.py`: ViFi-CLIP feature extractor loader
- `train.py`: training loop, validation, seed setting
- `clip/`: vendored CLIP architecture (for weight compatibility)

## 1. Environment Setup

In [ ]:
%pip install ftfy regex decord scikit-learn kagglehub pandas

In [ ]:
import kagglehub

# Download and cache the dataset (returns the local path)
dataset_path = kagglehub.dataset_download('farhanwew/real-vs-gen-videos')
print(f'Dataset path: {dataset_path}')

## 2. Load Feature Extractor

In [ ]:
from config import CLIP_ARCH, CLASS_NAMES, VIFICLIP_CHECKPOINT
from encoder import load_feature_extractor

feature_extractor = load_feature_extractor(
    arch=CLIP_ARCH,
    class_names=CLASS_NAMES,
    checkpoint_path=VIFICLIP_CHECKPOINT,
).cuda()

## 3. Prepare Data

In [ ]:
import os
from config import DATASET_ROOT, METADATA_CSV, BATCH_SIZE, VAL_SPLIT, NUM_WORKERS
from model import build_transform
from dataset import get_train_val_loaders

preprocess = build_transform(224)
train_loader, val_loader = get_train_val_loaders(
    transform=preprocess,
    dataset_root=DATASET_ROOT,
    metadata_csv=METADATA_CSV,
    val_split=VAL_SPLIT,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
)

## 4. Train

In [ ]:
from config import LEARNING_RATE, NUM_EPOCHS, INPUT_DIM, NUM_CLASSES, BEST_MODEL_SAVE_PATH
from train import set_seed, run_training

set_seed(42)

classifier = run_training(
    feature_extractor=feature_extractor,
    train_loader=train_loader,
    val_loader=val_loader,
    input_dim=INPUT_DIM,
    num_classes=NUM_CLASSES,
    lr=LEARNING_RATE,
    num_epochs=NUM_EPOCHS,
    save_path=BEST_MODEL_SAVE_PATH,
)

## 5. Save & Cleanup

In [ ]:
import shutil
from datetime import datetime

model_name = f"best_detector_model_{datetime.now().strftime('%Y%m%d_%H%M%S')}.pt"
shutil.copy(BEST_MODEL_SAVE_PATH, model_name)
shutil.copy(model_name, '/content/drive/MyDrive/')
print(f'Saved {model_name} to Drive')

In [ ]:
from google.colab import drive, runtime

drive.flush_and_unmount()
runtime.unassign()